# 2.0 Structure

Comprueba la etapa `structure` sobre la partición `train`.
Hallazgos que aplica: H-co-001 a H-co-004.

In [1]:
from mibici.data_preparation import structure
from mibici.pipeline import load_stage
import pandas as pd

raw = load_stage("raw")
structured = load_stage("structure")

raw_trips, raw_stations = raw["viajes"], raw["estaciones"]
trips, stations = structured["viajes"], structured["estaciones"]

2026-10-18 06:21:30.535 | INFO     | mibici.config:<module>:11 - PROJ_ROOT path is: X:\Repositorios\Personal\Kaggle\notebooks\mibici\mibici_predictions


2026-10-18 06:21:39.776 | INFO     | mibici.dataset:load_raw:166 - Partición train: 2024-10-01 a 2026-02-22
2026-10-18 06:21:39.785 | INFO     | mibici.dataset:load_raw:170 - Viajes cargados: 6,311,111


2026-10-18 06:21:49.001 | INFO     | mibici.dataset:load_raw:166 - Partición train: 2024-10-01 a 2026-02-22
2026-10-18 06:21:49.001 | INFO     | mibici.dataset:load_raw:170 - Viajes cargados: 6,311,111


## Filas y columnas

In [2]:
display(pd.DataFrame({
    "filas_raw": [len(raw_trips), len(raw_stations)],
    "filas_structure": [len(trips), len(stations)],
    "mismas_columnas": [
        raw_trips.columns.equals(trips.columns),
        raw_stations.columns.equals(stations.columns),
    ],
    "mismo_indice": [
        raw_trips.index.equals(trips.index),
        raw_stations.index.equals(stations.index),
    ],
}, index=["viajes", "estaciones"]))

,filas_raw,filas_structure,mismas_columnas,mismo_indice
viajes,6311111,6311111,True,True
estaciones,484,484,True,True


## Tipos

In [3]:
def compare_types(before, after):
    return pd.DataFrame({
        "raw": before.dtypes.astype(str),
        "structure": after.dtypes.astype(str),
    })

display(compare_types(raw_trips, trips))
display(compare_types(raw_stations, stations))

,raw,structure
Viaje_Id,int64,int64
Usuario_Id,int64,int64
Genero,object,string
Año_de_nacimiento,float64,Int64
Inicio_del_viaje,object,datetime64[ns]
Fin_del_viaje,object,datetime64[ns]
Origen_Id,int64,int64
Destino_Id,int64,int64


,raw,structure
id,int64,int64
name,object,string
latitude,float64,float64
longitude,float64,float64
dpcapacity,int64,int64


## Faltantes

In [4]:
display(pd.DataFrame({
    "raw": raw_trips.isna().sum(),
    "structure": trips.isna().sum(),
}))
display(pd.DataFrame({
    "raw": raw_stations.isna().sum(),
    "structure": stations.isna().sum(),
}))

,raw,structure
Viaje_Id,0,0
Usuario_Id,0,0
Genero,53941,53941
Año_de_nacimiento,1717,1717
Inicio_del_viaje,0,0
Fin_del_viaje,0,0
Origen_Id,0,0
Destino_Id,0,0


,raw,structure
id,0,0
name,0,0
latitude,0,0
longitude,0,0
dpcapacity,0,0


## Valores conservados

Las columnas convertidas deben conservar sus valores.
Las demás deben quedar idénticas.

In [5]:
# Año de nacimiento: mismo valor en las filas no faltantes
birth_raw = raw_trips["Año_de_nacimiento"]
birth_new = trips["Año_de_nacimiento"]
known = birth_raw.notna()
print("Año de nacimiento igual:", birth_raw[known].astype("int64").equals(
    birth_new[known].astype("int64")
))

# Género: mismas etiquetas y conteos
print("Género igual:", raw_trips["Genero"].astype("string").equals(trips["Genero"]))
display(trips["Genero"].value_counts(dropna=False))

# Nombre de estación: mismo texto
print("Nombre de estación igual:", raw_stations["name"].astype("string").equals(stations["name"]))

Año de nacimiento igual: True
Género igual: True


Genero
M       4528612
F       1728558
<NA>      53941
Name: count, dtype: Int64

Nombre de estación igual: True


In [6]:
# Fechas: se reconstruye el texto original a partir del valor convertido
def rebuild_text(dates, original):
    slash = original.str.contains("/", regex=False)
    iso = dates.dt.strftime("%Y-%m-%d %H:%M:%S")
    # El formato con diagonales no rellena la hora con cero ni incluye segundos
    day_first = (
        dates.dt.strftime("%d/%m/%Y ")
        + dates.dt.hour.astype(str)
        + dates.dt.strftime(":%M")
    )
    return iso.where(~slash, day_first)

for column in ["Inicio_del_viaje", "Fin_del_viaje"]:
    original = raw_trips[column].astype("string")
    rebuilt = rebuild_text(trips[column], original)
    differences = original.ne(rebuilt)
    print(f"{column}: NaT={trips[column].isna().sum():,}, "
          f"diferencias con el texto original={differences.sum():,}")
    display(pd.DataFrame({"original": original, "reconstruido": rebuilt})[differences].head())

Inicio_del_viaje: NaT=0, diferencias con el texto original=0


,original,reconstruido


Fin_del_viaje: NaT=0, diferencias con el texto original=0


,original,reconstruido


In [7]:
unchanged_trips = ["Viaje_Id", "Usuario_Id", "Origen_Id", "Destino_Id"]
unchanged_stations = ["id", "latitude", "longitude", "dpcapacity"]

display(pd.Series({
    **{f"viajes.{c}": raw_trips[c].equals(trips[c]) for c in unchanged_trips},
    **{f"estaciones.{c}": raw_stations[c].equals(stations[c]) for c in unchanged_stations},
}, name="identica"))

viajes.Viaje_Id          True
viajes.Usuario_Id        True
viajes.Origen_Id         True
viajes.Destino_Id        True
estaciones.id            True
estaciones.latitude      True
estaciones.longitude     True
estaciones.dpcapacity    True
Name: identica, dtype: bool

## Correspondencia entre funciones y etapa

In [8]:
# Aplicar las funciones individuales debe dar el mismo resultado que load_stage
manual_trips = structure.normalize_columns(raw_trips)
manual_trips = structure.convert_birth_year(manual_trips)
manual_trips = structure.convert_gender(manual_trips)
manual_trips = structure.convert_dates(manual_trips)

manual_stations = structure.normalize_columns(raw_stations)
manual_stations = structure.convert_station_name(manual_stations)

pd.testing.assert_frame_equal(manual_trips, trips)
pd.testing.assert_frame_equal(manual_stations, stations)
print("Funciones individuales = load_stage('structure')")

# Aplicar la etapa dos veces no debe cambiar el resultado
again = structure.run(structured)
pd.testing.assert_frame_equal(again["viajes"], trips)
pd.testing.assert_frame_equal(again["estaciones"], stations)
print("structure.run es idempotente")

# La etapa no modifica las tablas que recibe
print("raw sin cambios:", raw_trips["Inicio_del_viaje"].dtype == object
      and raw_trips["Genero"].dtype == object)

Funciones individuales = load_stage('structure')


structure.run es idempotente
raw sin cambios: True


## Conclusiones

Alcance: partición `train` (2024-10-01 a 2026-02-22), 6,311,111 viajes y 484 estaciones.

- Filas, columnas e índice se conservan en ambas tablas.
- Tipos: `Año_de_nacimiento` → Int64 (H-co-001), `Genero` → string (H-co-002),
  `Inicio_del_viaje` y `Fin_del_viaje` → datetime64[ns] (H-co-003), `name` → string (H-co-004).
  Los demás tipos no cambian.
- Faltantes iguales antes y después: 1,717 en `Año_de_nacimiento`, 53,941 en `Genero`
  y 0 en las demás columnas.
- Año, género y nombre conservan sus valores. Las fechas reconstruyen exactamente
  el texto original en ambos formatos (0 diferencias), lo que descarta intercambios
  de día y mes. Las columnas no convertidas quedan idénticas.
- Las funciones individuales producen el mismo resultado que `load_stage("structure")`;
  la etapa es idempotente y no modifica las tablas que recibe.

La etapa `structure` solo cambia tipos; no elimina ni modifica datos.